# Data Cleaning & Feature Engineering

## RabTech Academy — Data Analytics & Business Intelligence

### Submission 03

**Objective**

This project focuses on cleaning a messy payment transaction dataset and performing feature engineering to prepare the data for analysis.

The workflow includes:

- Loading and inspecting the raw dataset
- Identifying missing values and duplicate records
- Cleaning and standardizing data
- Handling missing numerical and categorical values
- Creating meaningful analytical features
- Validating the cleaned dataset
- Exporting the final dataset

In [1]:
import pandas as pd
import numpy as np

## 1. Load the Raw Dataset

The original messy payment dataset is loaded into a Pandas DataFrame.

The raw dataset is preserved without modification so that the effects of the cleaning process can be evaluated later.

In [2]:
df_raw = pd.read_csv(
    r"E:\Intern\My submissions\Submission 03\raw_messy_created_dataset.csv"
)

df_raw.head()

,payment_id,order_id,payment_date,payment_method,payment_status,amount_paid,transaction_fee,refund_amount
0,PAY1000000,ORD1000000,NaN,EMI,Failed,0.0,0.0,0.0
1,PAY1000001,ORD1000001,2/12/2023,UPI,Success,4998.0,0.0,0.0
2,PAY1000002,ORD1000002,12/14/2024,Net Banking,Success,867.3,0.0,0.0
3,PAY1000003,ORD1000003,NaN,Cash on Delivery,Not Charged,0.0,0.0,0.0
4,NaN,ORD1000004,7/27/2025,NaN,Not Charged,0.0,0.0,NaN


## 2. Raw Dataset Overview

Before performing any cleaning operation, the structure and size of the raw dataset are examined.

This initial inspection helps establish the baseline number of records and variables before data cleaning and transformation.

In [3]:
print("=" * 70)
print("RAW DATASET OVERVIEW")
print(f"Total Rows: {len(df_raw):,}")
print(f"Total Columns: {len(df_raw.columns)}")

RAW DATASET OVERVIEW
Total Rows: 115,360
Total Columns: 8


In [5]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 115360 entries, 0 to 115359
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   payment_id       103877 non-null  str    
 1   order_id         113055 non-null  str    
 2   payment_date     92286 non-null   str    
 3   payment_method   69196 non-null   str    
 4   payment_status   101562 non-null  str    
 5   amount_paid      101519 non-null  float64
 6   transaction_fee  106103 non-null  float64
 7   refund_amount    92251 non-null   float64
dtypes: float64(3), str(5)
memory usage: 7.0 MB


In [7]:
print(df_raw.isnull().sum())

payment_id         11483
order_id            2305
payment_date       23074
payment_method     46164
payment_status     13798
amount_paid        13841
transaction_fee     9257
refund_amount      23109
dtype: int64


## 3. Missing Value Audit

Missing values are identified before cleaning so that appropriate treatment can be applied to each variable.

Different columns may require different strategies depending on their data type and role in the dataset.

In [8]:
missing_audit = pd.DataFrame({
    "Missing Values": df_raw.isnull().sum(),
    "Missing Percentage": (
        df_raw.isnull().sum() / len(df_raw) * 100
    ).round(2)
})

missing_audit

,Missing Values,Missing Percentage
payment_id,11483,9.95
order_id,2305,2.00
payment_date,23074,20.00
payment_method,46164,40.02
payment_status,13798,11.96
amount_paid,13841,12.00
transaction_fee,9257,8.02
refund_amount,23109,20.03


## 4. Duplicate Record Audit

Duplicate records are checked at two levels:

1. **Exact duplicate rows** — records where every column contains the same value.
2. **Duplicate payment IDs** — repeated values in the primary key field.

The `payment_id` is treated as the primary identifier for a payment transaction.

In [12]:
print(f"Exact Duplicate Rows: {df_raw.duplicated().sum():,}")
print(f"Duplicate Payment IDs: {df_raw['payment_id'].duplicated().sum():,}")

Exact Duplicate Rows: 15,361
Duplicate Payment IDs: 25,359


In [15]:
df_clean = df_raw.copy()

## 5. Data Cleaning

A separate working copy of the raw dataset is created before applying any cleaning operations.

This preserves the original dataset for comparison and ensures that the raw data remains unchanged.

In [16]:
df_clean = df_raw.copy()

print(f"Raw records: {len(df_raw):,}")
print(f"Working records: {len(df_clean):,}")

Raw records: 115,360
Working records: 115,360


In [17]:
df_clean = df_clean.dropna(subset=['payment_id'])

df_clean = df_clean.drop_duplicates(
    subset=['payment_id'],
    keep='first'
)

### 5.1 Handle Missing and Duplicate Primary Keys

The `payment_id` column is treated as the primary identifier for each payment transaction.

Records with a missing `payment_id` are removed because they cannot be reliably identified.

Duplicate `payment_id` values are also removed, keeping the first occurrence to maintain a unique transaction identifier.

In [18]:
df_clean = df_clean.dropna(subset=['payment_id'])

df_clean = df_clean.drop_duplicates(
    subset=['payment_id'],
    keep='first'
)

print(f"Records after primary-key cleaning: {len(df_clean):,}")
print(f"Duplicate Payment IDs remaining: {df_clean['payment_id'].duplicated().sum():,}")

Records after primary-key cleaning: 90,000
Duplicate Payment IDs remaining: 0


In [19]:
df_clean['order_id'] = df_clean['order_id'].fillna('ORD_UNKNOWN')

### 5.2 Handle Missing Order IDs

Missing order identifiers are replaced with the standardized value `ORD_UNKNOWN`.

Unlike `payment_id`, the `order_id` is not used as the primary key for this dataset, so records with missing order IDs are retained rather than removed.

In [20]:
print(f"Missing order IDs before imputation: {df_clean['order_id'].isnull().sum():,}")

df_clean['order_id'] = df_clean['order_id'].fillna('ORD_UNKNOWN')

print(f"Missing order IDs after imputation: {df_clean['order_id'].isnull().sum():,}")

Missing order IDs before imputation: 0
Missing order IDs after imputation: 0


In [21]:
df_clean['payment_date'] = pd.to_datetime(
    df_clean['payment_date'],
    errors='coerce'
)

df_clean['payment_date'] = df_clean['payment_date'].ffill().bfill()

### 5.3 Handle and Standardize Payment Dates

The `payment_date` column is converted from its original mixed or inconsistent format into a standardized Pandas datetime format.

Invalid or unparseable date values are converted to `NaT` using `errors='coerce'`. Any resulting missing dates are then imputed using forward-fill followed by backward-fill to ensure that the dataset contains a complete payment date field.

In [22]:
print(f"Missing/invalid payment dates before cleaning: {df_clean['payment_date'].isnull().sum():,}")

df_clean['payment_date'] = pd.to_datetime(
    df_clean['payment_date'],
    errors='coerce'
)

print(f"Invalid dates after datetime conversion: {df_clean['payment_date'].isnull().sum():,}")

df_clean['payment_date'] = df_clean['payment_date'].ffill().bfill()

print(f"Missing payment dates after imputation: {df_clean['payment_date'].isnull().sum():,}")
print(f"Payment date data type: {df_clean['payment_date'].dtype}")

Missing/invalid payment dates before cleaning: 0
Invalid dates after datetime conversion: 0
Missing payment dates after imputation: 0
Payment date data type: datetime64[us]


In [23]:
df_clean['payment_method'] = df_clean['payment_method'].fillna('Unknown').str.strip().str.title()

df_clean['payment_status'] = df_clean['payment_status'].fillna('Pending').str.strip().str.title()

### 5.4 Clean and Standardize Categorical Columns

The `payment_method` and `payment_status` columns are cleaned and standardized.

Missing payment methods are replaced with `Unknown`, while missing payment statuses are replaced with `Pending`.

Leading and trailing whitespace is removed, and categorical values are converted to title case to maintain consistent category representation.

In [24]:
print("Missing categorical values before cleaning:")
print(f"Payment methods: {df_clean['payment_method'].isnull().sum():,}")
print(f"Payment statuses: {df_clean['payment_status'].isnull().sum():,}")

df_clean['payment_method'] = (
    df_clean['payment_method']
    .fillna('Unknown')
    .str.strip()
    .str.title()
)

df_clean['payment_status'] = (
    df_clean['payment_status']
    .fillna('Pending')
    .str.strip()
    .str.title()
)

print("\nMissing categorical values after cleaning:")
print(f"Payment methods: {df_clean['payment_method'].isnull().sum():,}")
print(f"Payment statuses: {df_clean['payment_status'].isnull().sum():,}")

Missing categorical values before cleaning:
Payment methods: 0
Payment statuses: 0

Missing categorical values after cleaning:
Payment methods: 0
Payment statuses: 0


In [25]:
print("Unique Payment Methods:")
print(df_clean['payment_method'].value_counts())

print("\n" + "-" * 50)

print("Unique Payment Statuses:")
print(df_clean['payment_status'].value_counts())

Unique Payment Methods:
payment_method
Unknown             36028
Upi                 17283
Cash On Delivery     9338
Credit Card          8128
Debit Card           6889
Net Banking          4474
Wallet               3690
Emi                  2111
Pay Later             981
Gift Card             567
Cardless Emi          511
Name: count, dtype: int64

--------------------------------------------------
Unique Payment Statuses:
payment_status
Success        65072
Pending        10951
Refunded        8475
Failed          4007
Not Charged     1495
Name: count, dtype: int64


In [26]:
for col in ['amount_paid', 'transaction_fee', 'refund_amount']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

median_amount = df_clean['amount_paid'].median()

df_clean['amount_paid'] = df_clean['amount_paid'].fillna(median_amount)
df_clean['transaction_fee'] = df_clean['transaction_fee'].fillna(0.0)
df_clean['refund_amount'] = df_clean['refund_amount'].fillna(0.0)

In [ ]:
df_clean['amount_paid'] = pd.to_numeric(df_clean['amount_paid'], errors='coerce')
df_clean['transaction_fee'] = pd.to_numeric(df_clean['transaction_fee'], errors='coerce')
df_clean['refund_amount'] = pd.to_numeric(df_clean['refund_amount'], errors='coerce')



### 5.5 Convert and Impute Numerical Fields

The numerical fields `amount_paid`, `transaction_fee`, and `refund_amount` are converted to numeric data types using `pd.to_numeric()`.

Values that cannot be interpreted as numbers are converted to missing values.

Missing `amount_paid` values are imputed using the median transaction amount because the median is less sensitive to extreme transaction values. Missing `transaction_fee` and `refund_amount` values are replaced with `0.0`, representing no recorded fee or refund.

In [30]:
print("Missing values before numerical cleaning:")

for col in ['amount_paid', 'transaction_fee', 'refund_amount']:
    print(f"{col}: {df_clean[col].isnull().sum():,}")

for col in ['amount_paid', 'transaction_fee', 'refund_amount']:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors='coerce'
    )

median_amount = df_clean['amount_paid'].median()

df_clean['amount_paid'] = df_clean['amount_paid'].fillna(median_amount)

df_clean['transaction_fee'] = df_clean['transaction_fee'].fillna(0.0)

df_clean['refund_amount'] = df_clean['refund_amount'].fillna(0.0)

print("\nMissing values after numerical cleaning:")

for col in ['amount_paid', 'transaction_fee', 'refund_amount']:
    print(f"{col}: {df_clean[col].isnull().sum():,}")

Missing values before numerical cleaning:
amount_paid: 0
transaction_fee: 0
refund_amount: 0

Missing values after numerical cleaning:
amount_paid: 0
transaction_fee: 0
refund_amount: 0


In [31]:
print("Numerical column data types:")
print(df_clean[['amount_paid', 'transaction_fee', 'refund_amount']].dtypes)

Numerical column data types:
amount_paid        float64
transaction_fee    float64
refund_amount      float64
dtype: object


In [32]:
print("=" * 70)

print("\n                  POST-CLEANING AUDIT                  ")

print(f"Clean Records Retained: {len(df_clean):,}")

print("Remaining Missing Values:")

print(df_clean.isnull().sum())


                  POST-CLEANING AUDIT                  
Clean Records Retained: 90,000
Remaining Missing Values:
payment_id         0
order_id           0
payment_date       0
payment_method     0
payment_status     0
amount_paid        0
transaction_fee    0
refund_amount      0
dtype: int64


### 5.6 Post-Cleaning Data Quality Audit

A post-cleaning audit is performed to verify the effectiveness of the data-cleaning operations.

The audit checks the number of records retained and identifies any remaining missing values across all columns. This provides evidence that the cleaning rules have been successfully applied before the dataset is used for feature engineering and export.

In [33]:
print("=" * 70)
print("                 POST-CLEANING AUDIT")
print("=" * 70)

print(f"Clean Records Retained: {len(df_clean):,}")
print(f"Total Columns: {len(df_clean.columns)}")

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum())

print("\nTotal Remaining Missing Values:")
print(df_clean.isnull().sum().sum())

                 POST-CLEANING AUDIT
Clean Records Retained: 90,000
Total Columns: 8

Remaining Missing Values:
payment_id         0
order_id           0
payment_date       0
payment_method     0
payment_status     0
amount_paid        0
transaction_fee    0
refund_amount      0
dtype: int64

Total Remaining Missing Values:
0


In [34]:
print("Primary Key Validation")
print("-" * 40)

print(f"Missing Payment IDs: {df_clean['payment_id'].isnull().sum():,}")
print(f"Duplicate Payment IDs: {df_clean['payment_id'].duplicated().sum():,}")

Primary Key Validation
----------------------------------------
Missing Payment IDs: 0
Duplicate Payment IDs: 0


In [35]:
df_clean.head()

,payment_id,order_id,payment_date,payment_method,payment_status,amount_paid,transaction_fee,refund_amount
0,PAY1000000,ORD1000000,2023-02-12,Emi,Failed,0.00,0.0,0.0
1,PAY1000001,ORD1000001,2023-02-12,Upi,Success,4998.00,0.0,0.0
2,PAY1000002,ORD1000002,2024-12-14,Net Banking,Success,867.30,0.0,0.0
3,PAY1000003,ORD1000003,2024-12-14,Cash On Delivery,Not Charged,0.00,0.0,0.0
5,PAY1000005,ORD1000005,2025-01-08,Cash On Delivery,Success,8275.89,0.0,0.0


## 6. Feature Engineering

Feature engineering is performed on the cleaned dataset to create additional variables that support financial and temporal analysis.

The payment date is transformed into month, month name, year, and quarter attributes. Financial features are also derived to calculate the net settlement amount, retention margin percentage, and profitability status for each transaction.

In [36]:
df_clean['order_month'] = df_clean['payment_date'].dt.month

df_clean['order_month_name'] = df_clean['payment_date'].dt.strftime('%B')

df_clean['order_year'] = df_clean['payment_date'].dt.year

df_clean['quarter'] = df_clean['payment_date'].dt.to_period('Q').astype(str)

In [37]:
df_clean['net_settlement_amount'] = (
    df_clean['amount_paid']
    - df_clean['transaction_fee']
    - df_clean['refund_amount']
)

df_clean['retention_margin_pct'] = np.where(
    df_clean['amount_paid'] > 0,
    (df_clean['net_settlement_amount'] / df_clean['amount_paid']) * 100,
    0.0
)

df_clean['profitability_status'] = np.where(
    df_clean['net_settlement_amount'] > 0,
    'Profitable',
    np.where(
        df_clean['net_settlement_amount'] == 0,
        'Break-Even',
        'Loss'
    )
)

In [38]:
print("=" * 110)
print("                 FEATURE ENGINEERING COMPLETE")
print("=" * 110)

print(
    df_clean[
        [
            'payment_id',
            'order_month_name',
            'order_year',
            'net_settlement_amount',
            'retention_margin_pct',
            'profitability_status'
        ]
    ].head()
)

                 FEATURE ENGINEERING COMPLETE
   payment_id order_month_name  order_year  net_settlement_amount  \
0  PAY1000000         February        2023                   0.00   
1  PAY1000001         February        2023                4998.00   
2  PAY1000002         December        2024                 867.30   
3  PAY1000003         December        2024                   0.00   
5  PAY1000005          January        2025                8275.89   

   retention_margin_pct profitability_status  
0                   0.0           Break-Even  
1                 100.0           Profitable  
2                 100.0           Profitable  
3                   0.0           Break-Even  
5                 100.0           Profitable  


### 6.1 Validate Engineered Features

The newly created features are validated before final export.

The validation checks whether the engineered columns are present, whether the financial calculations contain missing values, and whether the profitability categories have been generated correctly.

In [39]:
engineered_columns = [
    'order_month',
    'order_month_name',
    'order_year',
    'quarter',
    'net_settlement_amount',
    'retention_margin_pct',
    'profitability_status'
]

print("Engineered Columns:")
print(engineered_columns)

print("\nMissing Values in Engineered Features:")
print(df_clean[engineered_columns].isnull().sum())

print("\nProfitability Status Distribution:")
print(df_clean['profitability_status'].value_counts())

print("\nNet Settlement Summary:")
print(df_clean['net_settlement_amount'].describe())

Engineered Columns:
['order_month', 'order_month_name', 'order_year', 'quarter', 'net_settlement_amount', 'retention_margin_pct', 'profitability_status']

Missing Values in Engineered Features:
order_month              0
order_month_name         0
order_year               0
quarter                  0
net_settlement_amount    0
retention_margin_pct     0
profitability_status     0
dtype: int64

Profitability Status Distribution:
profitability_status
Profitable    77071
Break-Even    10565
Loss           2364
Name: count, dtype: int64

Net Settlement Summary:
count     90000.000000
mean      19844.409864
std       40484.776585
min     -317282.330000
25%        2000.067500
50%        8163.545000
75%       17559.752500
max      607058.550000
Name: net_settlement_amount, dtype: float64


In [40]:
print("Financial Logic Validation")
print("-" * 50)

expected_net = (
    df_clean['amount_paid']
    - df_clean['transaction_fee']
    - df_clean['refund_amount']
)

print(
    "Incorrect Net Settlement Calculations:",
    (df_clean['net_settlement_amount'] != expected_net).sum()
)

expected_margin = np.where(
    df_clean['amount_paid'] > 0,
    (df_clean['net_settlement_amount'] / df_clean['amount_paid']) * 100,
    0.0
)

print(
    "Incorrect Retention Margin Calculations:",
    (~np.isclose(
        df_clean['retention_margin_pct'],
        expected_margin
    )).sum()
)

Financial Logic Validation
--------------------------------------------------
Incorrect Net Settlement Calculations: 0
Incorrect Retention Margin Calculations: 0


## 7. Data Cleaning Report Export

The raw dataset, cleaned standardized dataset, and data-quality audit summary are exported into a single Excel workbook.

The workbook provides a transparent record of the cleaning process and allows the original and cleaned datasets to be compared alongside the cleaning audit results.

In [41]:
audit_summary = pd.DataFrame({

    'Metric': [
        'Total Raw Records',
        'Missing Primary Key (payment_id)',
        'Duplicate Records Removed',
        'Missing Dates Fixed',
        'Missing Methods Imputed',
        'Missing Status Imputed',
        'Missing Amounts Imputed',
        'Final Clean Records'
    ],

    'Count': [
        len(df_raw),
        df_raw['payment_id'].isnull().sum(),
        df_raw['payment_id'].duplicated().sum(),
        df_raw['payment_date'].isnull().sum(),
        df_raw['payment_method'].isnull().sum(),
        df_raw['payment_status'].isnull().sum(),
        df_raw['amount_paid'].isnull().sum(),
        len(df_clean)
    ]
})

audit_summary

,Metric,Count
0,Total Raw Records,115360
1,Missing Primary Key (payment_id),11483
2,Duplicate Records Removed,25359
3,Missing Dates Fixed,23074
4,Missing Methods Imputed,46164
5,Missing Status Imputed,13798
6,Missing Amounts Imputed,13841
7,Final Clean Records,90000


In [42]:
df_raw['payment_id'].isnull().sum()

np.int64(11483)

In [44]:
df_raw['amount_paid'].isnull().sum()

np.int64(13841)

In [45]:
with pd.ExcelWriter(
    'Data_Cleaning_Report_Task3.xlsx',
    engine='openpyxl'
) as writer:

    df_raw.to_excel(
        writer,
        sheet_name='Raw Messy Data',
        index=False
    )

    df_clean.to_excel(
        writer,
        sheet_name='Clean Standardized Data',
        index=False
    )

    audit_summary.to_excel(
        writer,
        sheet_name='Data Audit Summary',
        index=False
    )

print("\n" + "*" * 70)
print("Excel export complete: 'Data_Cleaning_Report_Task3.xlsx'")
print("*" * 70)


**********************************************************************
Excel export complete: 'Data_Cleaning_Report_Task3.xlsx'
**********************************************************************


In [46]:
df_clean.to_csv('clean_dataset.csv', index=False)

## 8. Final Clean Dataset Export

The fully cleaned and feature-engineered dataset is exported as `clean_dataset.csv`.

This file represents the final analytical dataset after data cleaning, standardization, imputation, validation, and feature engineering.

In [47]:
df_clean.to_csv('clean_dataset.csv', index=False)

print("\n" + "*" * 110)
print("Assignment Step Completed: Exported clean_dataset.csv")
print("=" * 110)


**************************************************************************************************************
Assignment Step Completed: Exported clean_dataset.csv


### 8.1 Final Dataset Verification

The final exported dataset is verified for record count, column count, missing values, duplicate primary keys, and the presence of engineered features.

In [48]:
print("=" * 70)
print("                 FINAL DATASET VERIFICATION")
print("=" * 70)

print(f"Final Records: {len(df_clean):,}")
print(f"Final Columns: {len(df_clean.columns):,}")

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sum())

print("\nDuplicate Payment IDs:")
print(df_clean['payment_id'].duplicated().sum())

print("\nFinal Columns:")
print(df_clean.columns.tolist())

print("\nFirst 5 Records:")
display(df_clean.head())

                 FINAL DATASET VERIFICATION
Final Records: 90,000
Final Columns: 15

Remaining Missing Values:
0

Duplicate Payment IDs:
0

Final Columns:
['payment_id', 'order_id', 'payment_date', 'payment_method', 'payment_status', 'amount_paid', 'transaction_fee', 'refund_amount', 'order_month', 'order_month_name', 'order_year', 'quarter', 'net_settlement_amount', 'retention_margin_pct', 'profitability_status']

First 5 Records:


,payment_id,order_id,payment_date,payment_method,payment_status,amount_paid,transaction_fee,refund_amount,order_month,order_month_name,order_year,quarter,net_settlement_amount,retention_margin_pct,profitability_status
0,PAY1000000,ORD1000000,2023-02-12,Emi,Failed,0.00,0.0,0.0,2,February,2023,2023Q1,0.00,0.0,Break-Even
1,PAY1000001,ORD1000001,2023-02-12,Upi,Success,4998.00,0.0,0.0,2,February,2023,2023Q1,4998.00,100.0,Profitable
2,PAY1000002,ORD1000002,2024-12-14,Net Banking,Success,867.30,0.0,0.0,12,December,2024,2024Q4,867.30,100.0,Profitable
3,PAY1000003,ORD1000003,2024-12-14,Cash On Delivery,Not Charged,0.00,0.0,0.0,12,December,2024,2024Q4,0.00,0.0,Break-Even
5,PAY1000005,ORD1000005,2025-01-08,Cash On Delivery,Success,8275.89,0.0,0.0,1,January,2025,2025Q1,8275.89,100.0,Profitable
